# 问题三方法优化探索

本 Notebook 复现 `src/experimental/q3_method_search.py` 的核心流程。目标不是直接证明因果，而是在描述性分析、控制变量回归、稳健性检查和机器学习特征重要性之间比较哪种方法更适合作为问题三主分析。

## 1. 读取数据与构造变量

外部变量来自 `modeling_base_table.csv`。天气、节假日、活动日是日期层面变量；销量是门店-商品层面变量，因此要特别注意同一天外部变量被重复用于多行样本。标准误需要按日期聚类。

In [1]:
from pathlib import Path
import sys
ROOT = Path.cwd()
while not (ROOT / 'AGENTS.md').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
from src.experimental.q3_method_search import prepare_data
analysis, daily, temp_bin_meta = prepare_data()
analysis.shape, daily.shape, temp_bin_meta


((59049, 56),
 (729, 16),
   temp_bin  avg_temperature_min  avg_temperature_max
 0       T1                 -2.5                 11.0
 1       T2                 11.0                 19.0
 2       T3                 19.0                 26.0
 3       T4                 26.0                 32.0)

## 2. 描述性分组比较

描述性均值用于观察现象，但没有控制门店、商品、星期和月份，因此不能作为因果结论。

In [2]:
from src.experimental.q3_method_search import group_binary_diff, holiday_weekend_crosstab
group_binary_diff(daily).head(20)


,factor,level,n_days,mean_daily_total_sales,median_daily_total_sales,std_daily_total_sales
0,天气合并类别,小雨/阵雨,281,137.544484,121.0,178.037734
1,天气合并类别,中大雨及以上,39,107.128205,96.0,60.917191
2,天气合并类别,无降水,406,124.283251,113.0,78.554918
3,天气合并类别,雨夹雪,3,356.666667,444.0,172.468355
4,是否节假日,0.0,669,128.452915,115.0,130.712303
5,是否节假日,1.0,60,140.366667,133.0,77.840333
6,是否周末,0.0,521,124.658349,112.0,139.910914
7,是否周末,1.0,208,141.394231,127.0,86.752624
8,是否活动日,0.0,705,124.119149,113.0,123.919776
9,是否活动日,1.0,24,285.541667,263.0,125.640612


In [3]:
holiday_weekend_crosstab(daily)


,holiday_weekend_group,n_days,mean_daily_total_sales,median_daily_total_sales
2,非节假日周末,184,141.451087,125.5
0,节假日周末,24,140.958333,153.5
1,节假日工作日,36,139.972222,127.5
3,非节假日工作日,485,123.521649,112.0


## 3. 控制变量和固定效应回归

主推荐模型使用合并天气类别、`log1p` 销量、门店/商品/星期/月固定效应，并加入只使用过去信息构造的 `lag_7` 与 `rolling_28_prev`。这样做的目的不是提高预测精度，而是降低基础需求差异和历史销量水平的干扰。

In [4]:
from src.experimental.q3_method_search import run_regression_suite
summary, coef_table, metric_table, iqr_map = run_regression_suite(analysis, daily)
summary


,method,nobs,r_squared,adj_r_squared,target_scale,note
0,原方案详细天气FE回归,59049,0.025615,0.024773,原始正向销量,标准误按日期聚类；系数用于统计关联解释。
1,合并天气_对数销量_历史控制FE回归,58482,0.377662,0.377194,log1p销量,标准误按日期聚类；系数用于统计关联解释。
2,温度分箱_对数销量_历史控制FE回归,58482,0.377658,0.377179,log1p销量,标准误按日期聚类；系数用于统计关联解释。
3,周末补充_无星期FE回归,58482,0.377546,0.377131,log1p销量,标准误按日期聚类；系数用于统计关联解释。
4,去异常日_对数销量_历史控制FE回归,56214,0.371873,0.371381,log1p销量；去日总销量异常日,标准误按日期聚类；系数用于统计关联解释。
5,标准化销量_历史控制FE回归,58482,0.040124,0.039401,门店-商品内标准化销量,标准误按日期聚类；系数用于统计关联解释。


In [5]:
coef_table[coef_table['model'].eq('合并天气_对数销量_历史控制FE回归')].head(20)


,model,target_scale,term,factor,variable,coef,std_err,p_value,direction,comparable_abs_effect
17,合并天气_对数销量_历史控制FE回归,log1p销量,is_activity_day,is_activity_day,is_activity_day,0.325608,0.041372,3.539354e-15,正关联,0.325608
18,合并天气_对数销量_历史控制FE回归,log1p销量,"C(weather_group, Treatment(reference='no_preci...",weather_group,雨夹雪,0.324700,0.078967,3.925344e-05,正关联,0.324700
19,合并天气_对数销量_历史控制FE回归,log1p销量,"C(weather_group, Treatment(reference='no_preci...",weather_group,中大雨及以上,-0.053247,0.023884,2.578636e-02,负关联,0.053247
20,合并天气_对数销量_历史控制FE回归,log1p销量,is_holiday,is_holiday,is_holiday,0.046345,0.019042,1.494063e-02,正关联,0.046345
21,合并天气_对数销量_历史控制FE回归,log1p销量,wind_power,wind_power,wind_power,-0.011509,0.004139,5.428085e-03,负关联,0.023017
22,合并天气_对数销量_历史控制FE回归,log1p销量,avg_temperature,avg_temperature,avg_temperature,0.001069,0.001554,4.914013e-01,正关联,0.016039
23,合并天气_对数销量_历史控制FE回归,log1p销量,temperature_range,temperature_range,temperature_range,-0.002332,0.002151,2.784087e-01,负关联,0.011660
24,合并天气_对数销量_历史控制FE回归,log1p销量,"C(weather_group, Treatment(reference='no_preci...",weather_group,小雨/阵雨,-0.006941,0.011997,5.628788e-01,负关联,0.006941


## 4. 稳健性检查

分门店、分类别和去除日总销量异常日后的结果用于判断结论是否稳定。若一个因素只在少数组别显著，论文中应降低结论强度。

In [6]:
from src.experimental.q3_method_search import robustness_by_group
store_robustness = robustness_by_group(analysis, 'store_id')
category_robustness = robustness_by_group(analysis, 'category')
store_robustness


,store_id,nobs,r_squared,activity_coef,activity_p,holiday_coef,holiday_p,avg_temperature_coef,avg_temperature_p,weather_max_abs_coef,weather_terms_count,status
0,14,9386,0.465156,0.415951,3.372111e-23,-0.007925,0.734232,0.006006,0.003977,0.258790,3,ok
1,18,7220,0.250767,0.062911,5.225021e-01,-0.043529,0.398259,0.000503,0.917929,0.688050,3,ok
2,23,8664,0.248933,0.443177,1.464899e-19,0.097093,0.006105,0.005815,0.006472,0.303307,3,ok
3,47,8664,0.395834,0.611937,4.982609e-15,0.064535,0.059062,-0.000172,0.950154,0.143651,3,ok
4,69,8664,0.478355,0.436400,1.467849e-09,0.195263,0.000004,-0.004157,0.128910,0.241776,3,ok
5,178,8664,0.569445,0.220391,1.547893e-02,-0.024688,0.511635,0.003211,0.340397,0.244382,3,ok
6,276,7220,0.147606,0.028319,7.102951e-02,-0.002976,0.811696,0.000509,0.626947,0.171558,3,ok


In [7]:
category_robustness


,category,nobs,r_squared,activity_coef,activity_p,holiday_coef,holiday_p,avg_temperature_coef,avg_temperature_p,weather_max_abs_coef,weather_terms_count,status
0,乳类饮品,7220,0.260148,0.205947,2.540369e-07,0.088202,0.001332,0.002621,0.233418,0.169166,3,ok
1,功能饮料,5054,0.379986,0.328149,1.365811e-03,0.081363,0.158763,0.006168,0.179770,0.354829,3,ok
2,包装冲饮,2166,0.088013,-0.013112,7.277915e-02,-0.015406,0.059504,0.000106,0.875886,0.033677,3,ok
3,包装坚果,5054,0.313787,0.363304,3.902569e-11,0.001308,0.966432,-0.003040,0.200239,0.453800,3,ok
4,包装散称,5054,0.414802,0.837852,1.417724e-15,0.004041,0.943660,-0.000582,0.889779,0.606534,3,ok
5,包装海苔,13718,0.339288,0.368311,1.888986e-16,0.064311,0.000915,-0.000134,0.921685,0.272407,3,ok
6,碳酸饮料,10108,0.453356,0.361443,1.121240e-25,0.063004,0.021906,0.009664,0.000113,0.308762,3,ok
7,速食类,10108,0.284744,0.163769,1.598243e-06,-0.011395,0.513454,0.000912,0.531884,0.029791,3,ok


## 5. 机器学习特征重要性

随机森林置换重要性用于判断非线性预测贡献。它不能说明方向，也不能证明因果，因此只适合作为辅助分析。

In [8]:
from src.experimental.q3_method_search import run_random_forest
rf_metrics, rf_importance = run_random_forest(analysis)
rf_metrics, rf_importance.head(15)


(            analysis_level       MAE      RMSE      WAPE  \
 0  row_level_store_product  1.892428  4.178186  0.846576   
 
                                    method  train_rows  validation_rows  
 0  RandomForest grouped weather + history       56052             2430  ,
               feature  importance_mean_mae_increase  importance_std
 7     rolling_28_prev                  1.420364e+00    4.099229e-02
 6               lag_7                  8.694374e-02    6.111175e-03
 5     is_activity_day                  7.382780e-02    1.116386e-02
 10     product_id_str                  2.640097e-02    3.439892e-03
 13       category_str                  1.453738e-02    3.936587e-03
 4          is_weekend                  6.546463e-03    3.985091e-03
 9        store_id_str                  5.557620e-03    4.842600e-03
 11        weekday_str                  3.325445e-03    1.726759e-03
 2          wind_power                  2.778241e-03    2.672667e-02
 0     avg_temperature               

## 6. 结论口径

本问更适合写成“统计关联分析”。若使用“影响”一词，应明确限定为观察数据中的条件关联，不写成“导致”。